# Cup-with-Handle t05 — 15-Year History Backtest — PIT variant C (causal-min handle)

Copy of `cup_handle_15yr_history_1.ipynb`; the original and the PIT notebook are untouched.
Every file this notebook writes carries a `_PIT_C` suffix. `handle_score_thresholds.json`
is read-only (OLD frozen edges); the in-sample re-fit in Cell 7 is computed in memory.

**Cell 3 change (detection only):** for each cup, take the FIRST close > rim after the
right rim (`j`). Handle low = `min(L[rch+1 : j])` — the lowest low strictly between the
right rim and the breakout bar, known at the close of `j`; no handle-pivot confirmation.
Windows: handle within 30 bars of the right rim, breakout within 15 bars of the handle low.
Same retracement (10–50%) and "above cup low" checks; same entry/stop/t05/v2b/non-overlap.
Causal: the right rim is a pivot high (max of rch±5), so no close in rch+1..rch+5 can beat
the rim — j >= rch+6 always, so both rims are confirmed by j.

Cell 7 reports ORIGINAL | PIT headline | PIT_C side by side (+0.05R cost, + IS-refit quintiles).
Cell 8 is live parity on the paper-arm window for all three. Run top to bottom.


In [ ]:
# ===== Cell 1 — Config (ISOLATED output) =====
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, glob, time, json
from pathlib import Path
from numba import njit
import warnings; warnings.filterwarnings('ignore')

DATA_DIR   = Path('/content/drive/MyDrive/Bukowski')          # your per-ticker price CSVs live here
OUT_DIR    = DATA_DIR / 'results' / 'history_15yr'            # ISOLATED — nothing else writes here
OUT_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLDS = DATA_DIR / 'results' / 'handle_score_thresholds.json'   # FROZEN — read only, never re-fit

# --- constants (identical to your v2b / scanner / exit-sweep) ---
ATR_PERIOD, MAX_STOP_PCT, MIN_STOP_ATR = 14, 0.10, 1.0
STOCK_FAST_SMA, STOCK_SLOW_SMA = 50, 200
TIME_STOP_DAYS = 120
T05_TARGET_MULT = 0.5           # t05 = breakout + 0.5 * cup_depth
IS_END = pd.Timestamp('2024-12-31')

# detector params — your scanner defaults, verbatim
DET = dict(pivot_left=5, pivot_right=5, min_cup_duration=30, max_cup_duration=130,
           min_cup_depth_pct=0.12, max_cup_depth_pct=0.50, rch_lch_tolerance=0.15,
           min_rounded_bars=3, rounded_band_pct=0.05, rounded_span_min=5,
           max_handle_duration=30, min_handle_retr_pct=0.10, max_handle_retr_pct=0.50)
print('Config loaded. Output ->', OUT_DIR)

# ---- PIT_C additions ----
CONFIRM_WINDOW_BARS = 15        # breakout within 15 bars of the handle low (matches detectFire)
PIT_TAG = '_PIT_C'              # every output file gets this suffix; originals untouched
ORIG_TRADES = OUT_DIR / 'trades_t05_15yr.csv'       # READ-ONLY: original run
PIT_TRADES  = OUT_DIR / 'trades_t05_15yr_PIT.csv'   # READ-ONLY: PIT headline (if it has been run)
COST_R = 0.05                   # assumed round-trip cost, in R
print('PIT_C mode | outputs suffixed', PIT_TAG)


In [ ]:
# ===== Cell 2 — Load 15y universe into a price DB (once) =====
def compute_atr(df, period=14):
    h,l,c = df['High'],df['Low'],df['Close']; pc=c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return tr.rolling(period).mean()

csv_files = sorted(glob.glob(f'{DATA_DIR}/*.csv'))
csv_files = [f for f in csv_files if Path(f).stem.upper() != 'SPY']
print(f'{len(csv_files)} ticker files found.')

price_db = {}                 # ticker -> dict of arrays for detection + sim
t0 = time.time()
for i,f in enumerate(csv_files):
    tk = Path(f).stem.upper()
    try:
        d = pd.read_csv(f, usecols=['Date','Open','High','Low','Close'])
        d['Date'] = pd.to_datetime(d['Date'])
        d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
        if len(d) < 260: continue
        d['ATR']  = compute_atr(d, ATR_PERIOD)
        d['S50']  = d['Close'].rolling(STOCK_FAST_SMA).mean()
        d['S200'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()
        price_db[tk] = dict(
            Date=d['Date'].values, Open=d['Open'].to_numpy(float), High=d['High'].to_numpy(float),
            Low=d['Low'].to_numpy(float), Close=d['Close'].to_numpy(float), ATR=d['ATR'].to_numpy(float),
            S50=d['S50'].to_numpy(float), S200=d['S200'].to_numpy(float),
            dates_ns=d['Date'].to_numpy('datetime64[ns]').astype(np.int64))
    except Exception:
        pass
    if (i+1) % 400 == 0: print(f'  loaded {len(price_db)} / {i+1} ({time.time()-t0:.0f}s)')
print(f'Universe: {len(price_db)} tickers with >=260 bars, in {time.time()-t0:.0f}s.')
rng = [pd.Timestamp(min(v["Date"][0]  for v in price_db.values())).date(),
       pd.Timestamp(max(v["Date"][-1] for v in price_db.values())).date()]
print('Price date range:', rng)

In [ ]:
# ===== Cell 3 — Historical fire detector — PIT variant C (causal-min handle) =====
# Your exact cup+handle geometry, adapted to scan FULL history and keep EVERY confirmed
# breakout as a fire (recency limit removed; entry = open on the bar after breakout).
def find_pivot_highs(H, left=5, right=5):
    n=len(H); out=[]
    for i in range(left, n-right):
        w=H[i-left:i+right+1]
        if H[i]==w.max() and (w==H[i]).sum()==1: out.append(i)
    return out
def find_pivot_lows(L, left=5, right=5):
    n=len(L); out=[]
    for i in range(left, n-right):
        w=L[i-left:i+right+1]
        if L[i]==w.min() and (w==L[i]).sum()==1: out.append(i)
    return out

def detect_fires_full(tk, p, det):
    O,H,L,C,ATR,D = p['Open'],p['High'],p['Low'],p['Close'],p['ATR'],p['Date']
    n=len(C)
    ph = find_pivot_highs(H, det['pivot_left'], det['pivot_right'])
    pl = find_pivot_lows(L,  det['pivot_left'], det['pivot_right'])
    if len(ph) < 2: return []
    fires=[]
    for a,lch in enumerate(ph[:-1]):
        lch_high = H[lch]
        for rch in ph[a+1:]:
            dur = rch-lch
            if dur < det['min_cup_duration']: continue
            if dur > det['max_cup_duration']: break
            rch_high = H[rch]
            if abs(rch_high-lch_high)/lch_high > det['rch_lch_tolerance']: continue
            cup_lows = L[lch+1:rch]
            if len(cup_lows) < 5: continue
            cup_low = cup_lows.min(); cup_low_idx = lch+1+int(cup_lows.argmin())
            cup_rim = (lch_high+rch_high)/2
            cup_depth_pct = (cup_rim-cup_low)/cup_rim
            if not (det['min_cup_depth_pct'] <= cup_depth_pct <= det['max_cup_depth_pct']): continue
            band = cup_low*(1+det['rounded_band_pct'])
            full = L[lch:rch+1]; inb=[k for k,v in enumerate(full) if v<=band]
            if len(inb) < det['min_rounded_bars']: continue
            if (max(inb)-min(inb)) < det['rounded_span_min']: continue
            cup_depth_abs = cup_rim-cup_low
            if cup_depth_abs <= 0: continue
            breakout = max(lch_high, rch_high)
            # VARIANT C: breakout bar j = FIRST close > rim after the right rim. Only the
            # first counts — if it fails a check below, this cup gets no trade.
            j_lim = min(rch + 1 + det['max_handle_duration'] + CONFIRM_WINDOW_BARS, n)
            confirm=None
            for j in range(rch+1, j_lim):
                if C[j] > breakout: confirm=j; break
            if confirm is None: continue
            # causal-min handle: lowest low strictly between the right rim and j
            seg = L[rch+1:confirm]
            if len(seg) == 0: continue
            h_idx = rch+1+int(seg.argmin()); handle_low = L[h_idx]
            if h_idx - rch > det['max_handle_duration']: continue      # handle within 30 bars of rim
            if confirm - h_idx > CONFIRM_WINDOW_BARS: continue         # breakout within 15 bars of handle
            retr = (rch_high-handle_low)/cup_depth_abs
            if not (det['min_handle_retr_pct'] <= retr <= det['max_handle_retr_pct']): continue
            if handle_low <= cup_low: continue
            if confirm is None or confirm+1 >= n: continue      # need next-day open to enter
            atr = ATR[rch]
            if np.isnan(atr) or atr<=0: continue
            e_idx = confirm+1
            e_px  = O[e_idx]
            raw_stop = handle_low*0.999
            stop_dist = e_px-raw_stop
            if stop_dist <= 0: continue
            if stop_dist < MIN_STOP_ATR*atr: stop_dist = MIN_STOP_ATR*atr
            if stop_dist/e_px > MAX_STOP_PCT: continue
            fires.append(dict(
                ticker=tk, entry_idx=e_idx,
                entry_date=pd.Timestamp(D[e_idx]), entry_price=e_px,
                stop_price=e_px-stop_dist, target_price=breakout+cup_depth_abs,
                breakout_level=breakout, atr=atr,
                handle_low_date=pd.Timestamp(D[h_idx]), rch_date=pd.Timestamp(D[rch]),
                cup_depth_pct=round(cup_depth_pct*100,1), handle_retr_pct=round(retr*100,1),
                # === the 3 frozen handle_score features ===
                days_since_handle_low=int(e_idx-h_idx),
                handle_dur_days=int((pd.Timestamp(D[h_idx])-pd.Timestamp(D[rch])).days),
                handle_depth_atr=(breakout-handle_low)/atr,
                confirm_idx=int(confirm)))
            break   # one cup per LCH (your logic)
    return fires

t0=time.time(); all_fires=[]
for i,(tk,p) in enumerate(price_db.items()):
    all_fires += detect_fires_full(tk,p,DET)
    if (i+1)%400==0: print(f'  scanned {i+1} tickers, {len(all_fires)} fires ({time.time()-t0:.0f}s)')
fires = pd.DataFrame(all_fires).sort_values('entry_date').reset_index(drop=True)
fires = fires.drop_duplicates(['ticker','handle_low_date']).reset_index(drop=True)
fires.to_csv(OUT_DIR/f'cup_handle_fires_15yr_raw{PIT_TAG}.csv', index=False)
print(f'\nRaw fires (pre-uptrend filter): {len(fires)}  |  {fires.entry_date.min().date()} -> {fires.entry_date.max().date()}  ({time.time()-t0:.0f}s)')
print('per year:', fires.entry_date.dt.year.value_counts().sort_index().to_dict())

In [ ]:
# ===== Cell 4 — v2b filter: stock 50>200 at entry (point-in-time, no lookahead) =====
def uptrend_at(tk, e_idx):
    p = price_db[tk]; s50,s200 = p['S50'][e_idx], p['S200'][e_idx]
    if np.isnan(s50) or np.isnan(s200): return False
    return s50 > s200
fires['stock_uptrend'] = [uptrend_at(t,i) for t,i in zip(fires.ticker, fires.entry_idx)]
v2b = fires[fires['stock_uptrend']].reset_index(drop=True)
print(f'After v2b uptrend filter: {len(v2b)} fires (from {len(fires)}).')
v2b.to_csv(OUT_DIR/f'cup_handle_v2b_fires_15yr{PIT_TAG}.csv', index=False)

In [ ]:
# ===== Cell 5 — t05 exit sim (your exact numba walk) + non-overlap =====
@njit(cache=True)
def _sim_trade(opens,highs,lows,closes, entry_idx, entry_price, stop_price,
               target_price, atr, time_stop_days):
    n=len(opens); end=entry_idx+time_stop_days
    if end>n: end=n
    risk = entry_price-stop_price
    for i in range(entry_idx, end):
        bl=lows[i]; bh=highs[i]
        if bl <= stop_price:
            return i, stop_price, 0, (stop_price-entry_price)/risk
        if bh >= target_price:
            return i, target_price, 1, (target_price-entry_price)/risk
    last=end-1
    return last, closes[last], 4, (closes[last]-entry_price)/risk

# warm up
_sim_trade(np.array([100.,101,102]),np.array([100.,101,102]),np.array([99.,100,101]),
           np.array([100.,101,102]),0,100.,95.,110.,2.,120)

REASON={0:'stop',1:'target',4:'time_stop'}
rows=[]
for _,f in v2b.iterrows():
    p=price_db[f['ticker']]; ei=int(f['entry_idx'])
    tgt = f['breakout_level'] + T05_TARGET_MULT*(f['target_price']-f['breakout_level'])   # t05 half target
    xi,xpx,rc,R = _sim_trade(p['Open'],p['High'],p['Low'],p['Close'],
                             ei, f['entry_price'], f['stop_price'], tgt, f['atr'], TIME_STOP_DAYS)
    rows.append(dict(ticker=f['ticker'], entry_date=f['entry_date'],
        exit_date=pd.Timestamp(p['Date'][xi]), entry=f['entry_price'], exit=xpx,
        stop_at_entry=f['stop_price'], R=R, exit_reason=REASON[rc],
        year=f['entry_date'].year,
        days_since_handle_low=f['days_since_handle_low'], handle_dur_days=f['handle_dur_days'],
        handle_depth_atr=f['handle_depth_atr'], bars_held=int(xi-ei)))
tr = pd.DataFrame(rows)

# non-overlap per ticker (your rule): skip a new entry until prior trade on same ticker exits
tr = tr.sort_values(['ticker','entry_date']).reset_index(drop=True)
keep=[]; last={}
for _,r in tr.iterrows():
    pv=last.get(r['ticker'])
    if pv is None or r['entry_date'] > pv:
        keep.append(True); last[r['ticker']]=r['exit_date']
    else: keep.append(False)
tr = tr[keep].sort_values('entry_date').reset_index(drop=True)
print(f'Trades after non-overlap: {len(tr)}  |  {tr.entry_date.min().date()} -> {tr.entry_date.max().date()}')

In [ ]:
# ===== Cell 6 — Frozen handle_score — OLD edges, NO re-freeze =====
thr = json.load(open(THRESHOLDS))
feats, hist, edges = thr['features'], thr['feature_hist'], thr['hscore_edges']
sorted_hist = {f: np.sort(np.array(hist[f], float)) for f in feats}
def pct_vs_frozen(f, x):                       # empirical percentile vs FROZEN history
    a = sorted_hist[f]; return np.searchsorted(a, x, side='right')/len(a)
sc = np.zeros(len(tr))
for f in feats:
    sc += (1 - tr[f].apply(lambda x: pct_vs_frozen(f, x)).values)
tr['hscore'] = sc/len(feats)
tr['q'] = pd.cut(tr['hscore'], bins=edges, labels=[1,2,3,4,5], include_lowest=True).astype('Int64')
tr['size_bucket'] = tr['q'].map({1:'skip',2:'skip',3:'half',4:'full',5:'full'})
tr.to_csv(OUT_DIR/f'trades_t05_15yr{PIT_TAG}.csv', index=False)
print(f'Saved trades_t05_15yr{PIT_TAG}.csv  |  n =', len(tr))

In [ ]:
# ===== Cell 7 — ORIGINAL | PIT headline | PIT_C, side by side =====
def pf(x):
    x=np.asarray(x,float); w=x[x>0].sum(); l=-x[x<0].sum(); return w/l if l>0 else np.nan

def _bars(r):
    d = price_db[r.ticker]['Date']
    return int(np.searchsorted(d, np.datetime64(r.exit_date)) - np.searchsorted(d, np.datetime64(r.entry_date)))
def load_trades(path):
    d = pd.read_csv(path, parse_dates=['entry_date','exit_date'])         # READ-ONLY
    if 'bars_held' not in d.columns: d['bars_held'] = [_bars(r) for r in d.itertuples()]
    d['year'] = d.entry_date.dt.year
    return d

SETS = {'ORIGINAL': load_trades(ORIG_TRADES)}
if PIT_TRADES.exists(): SETS['PIT headline'] = load_trades(PIT_TRADES)
else: print(f'NOTE: {PIT_TRADES.name} not found — run the PIT notebook first for that column.')
SETS['PIT_C'] = tr
STOP_FAST = 4   # stopped within 5 bars = stop exit on the entry bar .. 4 bars after it

def row(d):
    q35 = d[d.q.isin([3,4,5])]; IS = d[d.entry_date<=IS_END]; OOS = d[d.entry_date>IS_END]
    return dict(n=len(d), WR=(d.R>0).mean(), PF=pf(d.R), avgR=d.R.mean(),
                n_Q35=len(q35), WR_Q35=(q35.R>0).mean(), PF_Q35=pf(q35.R), avgR_Q35=q35.R.mean(),
                PF_Q35_cost=pf(q35.R-COST_R), avgR_Q35_cost=(q35.R-COST_R).mean(),
                n_IS=len(IS), PF_IS=pf(IS.R), n_OOS=len(OOS), PF_OOS=pf(OOS.R),
                pct_stop_le5bars=((d.exit_reason=='stop') & (d.bars_held<=STOP_FAST)).mean())
summ = pd.DataFrame({k: row(v) for k,v in SETS.items()}).T
pd.set_option('display.width', 220)
print(f'=== SUMMARY (IS <= {IS_END.date()} < OOS; *_cost = after {COST_R}R round trip) ===')
print(summ.to_string(float_format=lambda x: f'{x:.3f}'))

print('\n=== PER-YEAR PF (n) ===')
py = pd.DataFrame({k: v.groupby('year')['R'].apply(pf) for k,v in SETS.items()})
pn = pd.DataFrame({k+' n': v.groupby('year').size() for k,v in SETS.items()})
print(pd.concat([py, pn], axis=1).to_string(float_format=lambda x: f'{x:.2f}'))

def qtable(d, qcol, label):
    out=[]
    for q in [1,2,3,4,5]:
        s=d[d[qcol]==q]; a=s[s.year<2020]; b=s[s.year>=2020]
        out.append(dict(set=label, q=f'Q{q}', n=len(s), WR=(s.R>0).mean(), PF=pf(s.R), avgR=s.R.mean(),
                        n_pre2020=len(a), PF_pre2020=pf(a.R), n_2020p=len(b), PF_2020p=pf(b.R)))
    return out
print('\n=== QUINTILE PF — OLD frozen edges (all years | pre-2020 | 2020+) ===')
qt = pd.DataFrame([r for k,v in SETS.items() for r in qtable(v, 'q', k)])
print(qt.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# ---- IS-only RE-FIT on PIT_C: feature reference distributions AND quintile edges are
# ---- fit on entries <= IS_END, then applied unchanged to OOS (> IS_END). In memory only.
IS_C, OOS_C = tr[tr.entry_date<=IS_END], tr[tr.entry_date>IS_END]
ref = {f: np.sort(IS_C[f].astype(float).values) for f in feats}
def hs_refit(df):
    s = np.zeros(len(df))
    for f in feats:
        s += 1 - np.searchsorted(ref[f], df[f].astype(float).values, side='right')/len(ref[f])
    return s/len(feats)
is_hs = hs_refit(IS_C)
re_edges = np.unique(np.quantile(is_hs, [0,.2,.4,.6,.8,1]))
re_edges[0], re_edges[-1] = -np.inf, np.inf
labels = list(range(1, len(re_edges)))
trC = tr.copy(); trC['hs_re'] = hs_refit(trC)
trC['q_re'] = pd.cut(trC['hs_re'], bins=re_edges, labels=labels).astype('Int64')
print(f'\n=== PIT_C — quintiles RE-FIT on IS only (n_IS={len(IS_C)}), evaluated IS and OOS (n_OOS={len(OOS_C)}) ===')
print('re-fit hscore edges (IS quantiles):', np.round(np.quantile(is_hs, [.2,.4,.6,.8]), 3).tolist())
rq=[]
for q in labels:
    for lab, d in (('IS', trC[trC.entry_date<=IS_END]), ('OOS', trC[trC.entry_date>IS_END])):
        s=d[d.q_re==q]
        rq.append(dict(q=f'Q{q}', window=lab, n=len(s), WR=(s.R>0).mean(), PF=pf(s.R), avgR=s.R.mean(),
                       PF_cost=pf(s.R-COST_R)))
rq = pd.DataFrame(rq); print(rq.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
o35 = trC[(trC.entry_date>IS_END) & trC.q_re.isin([3,4,5])]
print(f'OOS Q3-Q5 (re-fit): n={len(o35)}  PF={pf(o35.R):.3f}  PF after {COST_R}R={pf(o35.R-COST_R):.3f}')

print('\n=== days_since_handle_low (bars, entry - handle low) ===')
for k,v in SETS.items():
    print(f'  {k:13s}', v.days_since_handle_low.describe()[['min','25%','50%','75%','max']].round(1).to_dict())

summ.to_csv(OUT_DIR/f'summary_15yr{PIT_TAG}.csv')
pd.concat([py, pn], axis=1).to_csv(OUT_DIR/f'per_year_15yr{PIT_TAG}.csv')
qt.to_csv(OUT_DIR/f'quintile_15yr{PIT_TAG}.csv', index=False)
rq.to_csv(OUT_DIR/f'quintile_refit_IS_15yr{PIT_TAG}.csv', index=False)
print(f'\nSaved summary/per_year/quintile/quintile_refit_IS _15yr{PIT_TAG}.csv')


In [ ]:
# ===== Cell 8 — LIVE PARITY on the PAPER-ARM window (ORIGINAL | PIT | PIT_C) =====
# Ticker + entry within 5 calendar days; window starts at the paper arm's first entry.
# Q3-Q5 by the OLD frozen edges (the tiers live actually used).
LP_DIR    = DATA_DIR / 'results' / 'live_parity'
PAPER_CSV = LP_DIR / 'paper.csv'
WIN_END, MATCH_DAYS = pd.Timestamp('2026-09-30'), 5

pp = pd.read_csv(PAPER_CSV)
pp['entry_dt'] = pd.to_datetime(pp['replay_entry_date'])
pp = pp[pp['replay_exit_reason'].isin(['stop','target','timeout'])]
WIN_START = pp['entry_dt'].min()
pp = pp[(pp.entry_dt >= WIN_START) & (pp.entry_dt <= WIN_END)].drop_duplicates(['ticker','entry_dt']).reset_index(drop=True)
print(f'paper-arm window: {WIN_START.date()} -> {WIN_END.date()}  |  paper closed trades: {len(pp)}')

def stats(r):
    r = pd.Series(r, dtype=float).dropna()
    if len(r)==0: return dict(n=0, WR=np.nan, PF=np.nan, avgR=np.nan, netR=0.0)
    return dict(n=len(r), WR=(r>0).mean(), PF=pf(r), avgR=r.mean(), netR=r.sum())

def parity(bt, label):
    bt = bt[bt.q.isin([3,4,5])]
    bt = bt[(bt.entry_date >= WIN_START) & (bt.entry_date <= WIN_END)].copy()
    open_end = bt.exit_reason.astype(str).str.contains('time') & (bt.exit_date >= WIN_END - pd.Timedelta(days=3))
    bt = bt[~open_end].reset_index(drop=True)
    pairs, used = [], set()
    for i,p in pp.iterrows():
        c = bt[(bt.ticker==p.ticker) & ((bt.entry_date-p.entry_dt).abs() <= pd.Timedelta(days=MATCH_DAYS))]
        c = c[~c.index.isin(used)]
        if len(c):
            j = (c.entry_date-p.entry_dt).abs().idxmin(); used.add(j); pairs.append((i,j))
    m = pd.DataFrame([dict(ticker=pp.at[i,'ticker'], entry_live=pp.at[i,'entry_dt'].date(),
                           entry_bt=bt.at[j,'entry_date'].date(), R_live_paper=pp.at[i,'replay_R'],
                           R_backtest=bt.at[j,'R']) for i,j in pairs])
    po = pp[~pp.index.isin({i for i,_ in pairs})]; bo = bt[~bt.index.isin(used)]
    agree = (np.sign(m.R_live_paper)==np.sign(m.R_backtest)).mean() if len(m) else np.nan
    print(f'\n--- {label}: backtest Q3-Q5 {len(bt)} | matched {len(m)} | paper-only {len(po)} | backtest-only {len(bo)} | sign agreement {agree:.0%}')
    print(pd.DataFrame({'backtest (all, window)': stats(bt.R), 'paper (all, window)': stats(pp.replay_R),
                        'matched - backtest R': stats(m.R_backtest if len(m) else []),
                        'paper-only': stats(po.replay_R), 'backtest-only': stats(bo.R)}).T
          .to_string(float_format=lambda x: f'{x:.3f}'))
    return m, po, bo

for k in [k for k in SETS if k != 'PIT_C']:
    parity(SETS[k], k)                                 # printed only — their files untouched
m, po, bo = parity(tr, 'PIT_C')
m.to_csv(LP_DIR/f'matched{PIT_TAG}.csv', index=False)
po.to_csv(LP_DIR/f'paper_only{PIT_TAG}.csv', index=False)
bo.to_csv(LP_DIR/f'backtest_only{PIT_TAG}.csv', index=False)
print(f'\nSaved matched/paper_only/backtest_only{PIT_TAG}.csv to {LP_DIR}')


In [ ]:
# ===== Cell 9 (optional, ~2-5 min) — CAUSALITY CHECK on real data =====
# For a random sample of PIT_C fires, re-run the detector on that ticker's data CUT at
# the entry bar. A causal detector must still produce the same entry. Any miss = lookahead.
rng = np.random.default_rng(0)
smp = fires.iloc[rng.choice(len(fires), size=min(100, len(fires)), replace=False)]
miss = []
for _, f in smp.iterrows():
    p = price_db[f.ticker]; e = int(f.entry_idx)
    q = {k: (v[:e+1] if isinstance(v, np.ndarray) else v) for k, v in p.items()}
    if not any(g['entry_idx'] == e for g in detect_fires_full(f.ticker, q, DET)):
        miss.append((f.ticker, f.entry_date.date()))
print(f'causality check: {len(smp)-len(miss)}/{len(smp)} fires reproduced on truncated data'
      + ('' if not miss else f'  LOOKAHEAD: {miss[:10]}'))
